# 01 - Bronze to Silver: types, nulls, quarantine, and deduplication

Prerequisites / execution context: Run 00 first. Uses only the dedicated kit schemas. Re-running replaces Silver exercise outputs.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R19: PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics); [R23: dropDuplicates](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/dropDuplicates); [R24: row_number](https://docs.databricks.com/aws/en/pyspark/reference/functions/row_number); [R25: NULL semantics](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-null-semantics).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Make invalid data observable
Use try_cast to identify malformed money without aborting the full batch. Treat missing customer IDs as a separate enrichment concern; the order itself can remain valid.


In [ ]:
raw = spark.table(f"{BRONZE}.orders_raw")
clean = (raw.withColumn("amount_decimal", F.expr("try_cast(amount AS DECIMAL(12,2))"))
 .withColumn("event_ts", F.expr("try_cast(event_time AS TIMESTAMP)"))
 .withColumn("customer_id", F.trim("customer_id"))
 .withColumn("quality_reason", F.when(F.col("amount_decimal").isNull(), "bad_amount")
             .when(F.col("amount_decimal") < 0, "negative_amount")
             .when(F.col("event_ts").isNull(), "bad_timestamp")
             .when(F.col("order_id").isNull(), "missing_order_id")))
quarantine = clean.filter("quality_reason IS NOT NULL")
valid = clean.filter("quality_reason IS NULL")
assert quarantine.count()==2
assert valid.count()==6
quarantine.write.mode("overwrite").saveAsTable(f"{SILVER}.orders_quarantine")


## 2. Choose the latest record deterministically
Remove exact duplicate events first. Then order by sequence, timestamp, and event ID. For this dataset the final tie breaker is unique after exact deduplication. A production contract must define a total ordering when conflicting payloads share all ordering values. Official Databricks documentation: [R24: row_number](https://docs.databricks.com/aws/en/pyspark/reference/functions/row_number).


In [ ]:
events = valid.dropDuplicates().drop("quality_reason", "amount", "event_time").withColumnRenamed("amount_decimal","amount")
w = Window.partitionBy("order_id").orderBy(F.desc("seq"),F.desc("event_ts"),F.desc("event_id"))
latest = events.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
assert events.count()==5
assert latest.count()==4
assert latest.filter("order_id='o1'").first().amount == __import__('decimal').Decimal('120.00')
latest.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.orders")
latest.orderBy("order_id").show(truncate=False)


## 3. Nulls and basic summaries
Count rows separately from non-null customer IDs. Use a null-safe comparison for migration mismatch logic. Official Databricks documentation: [R25: NULL semantics](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-null-semantics); [R74: DataFrame reference](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe).


In [ ]:
latest.select(F.count("*").alias("rows"),F.count("customer_id").alias("known_customer_ids"),
 F.approx_count_distinct("customer_id").alias("approx_customer_ids"),F.mean("amount").alias("mean_amount")).show()
latest.select("amount").summary("count","min","max","mean").show()
spark.sql("SELECT NULL = NULL AS ordinary, NULL <=> NULL AS null_safe").show()
assert latest.count()==4 and latest.filter("customer_id IS NOT NULL").count()==3


## Challenge and solution
Why does dropDuplicates(["order_id"]) not express the latest-record requirement? It does not encode which version to keep. Why is fillna(0) on money dangerous here? It would hide a malformed amount as a genuine zero. Reconciliation: 8 raw = 2 rejected + 1 exact duplicate removed + 1 older valid version removed + 4 current orders.
